In [1]:
import os, json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import sys
sys.path.append('..')
from src.evaluation import temporal_train_test_split, evaluate_model
from src.text_features import TextFeatureExtractor
from src.image_features import CLIPFeatureExtractor

MODEL_NAME = 'mmgcn_edl'
RESULTS_DIR = f'../results/{MODEL_NAME}'

# Arquitectura
EMBED_DIM = 64
DIM_LATENT_V = 256
AGGR_MODE = 'mean'
CONCATE = False
HAS_ID = True

# EDL
EVIDENCE_DIM = 32
LAMBDA_KL = 0.1
ANNEALING_STEP = 10

# Entrenamiento
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 1024
LIKE_THRESHOLD = 4.0
EPS = 1e-8

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


## Definición del modelo

**MMGCN (nb14) + EDL (nb12):** se mantiene la arquitectura fiel al repo original
(BaseModel + GCN por modalidad, 3 capas, LeakyReLU, preference, id\_embedding residual,
mini-batch BPR) y se reemplaza la fusión por promedio con la **capa de fusión evidencial**
de Lógica Subjetiva.

**Cambios respecto a nb14:**
- `Net.forward()` → fusión por `EvidentialFusionLayer` en vez de `(img + text) / 2`.
- `Net.loss()` → BPR + KL[Dir(α) ∥ Dir(1)] con annealing + L2 reg sobre embeddings.
- El loop de entrenamiento pasa `epoch` para el annealing del término KL.

In [2]:

class BaseModel(nn.Module):
    """
    Equivalente sin PyG a BaseModel(MessagePassing) del repo original.
    h = adj @ (x @ W),  con adj normalizada según aggr_mode.
    """
    def __init__(self, in_channels, out_channels, aggr='mean'):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(in_channels, out_channels))
        bound = 1.0 / math.sqrt(in_channels)
        nn.init.uniform_(self.weight, -bound, bound)

    def forward(self, x, adj):
        x = x @ self.weight
        return torch.sparse.mm(adj, x)


class GCN(nn.Module):
    """
    GCN por modalidad fiel al repo: 3 capas, preference learnable por usuario,
    id_embedding residual, F.normalize inicial, LeakyReLU en cada capa.
    """
    def __init__(self, num_user, num_item, dim_feat, dim_id,
                 aggr_mode, concate, has_id, dim_latent=None):
        super().__init__()
        self.dim_latent = dim_latent
        self.concate = concate
        self.has_id = has_id

        dim0 = dim_latent if dim_latent else dim_feat

        if dim_latent:
            self.preference = nn.Parameter(
                nn.init.xavier_normal_(torch.rand(num_user, dim_latent))
            )
            self.MLP = nn.Linear(dim_feat, dim_latent)
        else:
            self.preference = nn.Parameter(
                nn.init.xavier_normal_(torch.rand(num_user, dim_feat))
            )

        self.conv_embed_1 = BaseModel(dim0, dim0, aggr=aggr_mode)
        nn.init.xavier_normal_(self.conv_embed_1.weight)
        self.linear_layer1 = nn.Linear(dim0, dim_id)
        nn.init.xavier_normal_(self.linear_layer1.weight)
        self.g_layer1 = nn.Linear(dim0 + dim_id if concate else dim0, dim_id)
        nn.init.xavier_normal_(self.g_layer1.weight)

        self.conv_embed_2 = BaseModel(dim_id, dim_id, aggr=aggr_mode)
        nn.init.xavier_normal_(self.conv_embed_2.weight)
        self.linear_layer2 = nn.Linear(dim_id, dim_id)
        nn.init.xavier_normal_(self.linear_layer2.weight)
        self.g_layer2 = nn.Linear(dim_id * 2 if concate else dim_id, dim_id)
        nn.init.xavier_normal_(self.g_layer2.weight)

        self.conv_embed_3 = BaseModel(dim_id, dim_id, aggr=aggr_mode)
        nn.init.xavier_normal_(self.conv_embed_3.weight)
        self.linear_layer3 = nn.Linear(dim_id, dim_id)
        nn.init.xavier_normal_(self.linear_layer3.weight)
        self.g_layer3 = nn.Linear(dim_id * 2 if concate else dim_id, dim_id)
        nn.init.xavier_normal_(self.g_layer3.weight)

    def _layer(self, conv, linear, g, x, id_emb, adj):
        h = F.leaky_relu(conv(x, adj))
        x_hat = F.leaky_relu(linear(x))
        if self.has_id:
            x_hat = x_hat + id_emb
        if self.concate:
            return F.leaky_relu(g(torch.cat([h, x_hat], dim=1)))
        return F.leaky_relu(g(h) + x_hat)

    def forward(self, features, id_embedding, adj):
        temp = self.MLP(features) if self.dim_latent else features
        x = torch.cat([self.preference, temp], dim=0)
        x = F.normalize(x)
        x = self._layer(self.conv_embed_1, self.linear_layer1, self.g_layer1, x, id_embedding, adj)
        x = self._layer(self.conv_embed_2, self.linear_layer2, self.g_layer2, x, id_embedding, adj)
        x = self._layer(self.conv_embed_3, self.linear_layer3, self.g_layer3, x, id_embedding, adj)
        return x

In [3]:

class EvidentialFusionLayer(nn.Module):

    def __init__(self, input_dim: int, evidence_dim: int):
        super().__init__()
        self.K = evidence_dim
        self.proj_v = nn.Linear(input_dim, evidence_dim)
        self.proj_t = nn.Linear(input_dim, evidence_dim)

    def forward(self, e_v, e_t):
        evidence_v = F.softplus(self.proj_v(e_v))   # [N, K]
        evidence_t = F.softplus(self.proj_t(e_t))   # [N, K]
        alpha_v = evidence_v + 1.0
        alpha_t = evidence_t + 1.0
        S_v = alpha_v.sum(dim=-1, keepdim=True)
        S_t = alpha_t.sum(dim=-1, keepdim=True)
        u_v = self.K / (S_v + EPS)
        u_t = self.K / (S_t + EPS)
        exp_v = torch.exp(-u_v)
        exp_t = torch.exp(-u_t)
        w_v = exp_v / (exp_v + exp_t + EPS)
        w_t = exp_t / (exp_v + exp_t + EPS)
        e_final = w_v * e_v + w_t * e_t
        return e_final, alpha_v, alpha_t


class EvidentialLoss(nn.Module):
    """
    BPR + KL[Dir(alpha) || Dir(1,...,1)] con annealing lineal.
    """
    def __init__(self, lambda_kl: float = 0.1, annealing_step: int = 10):
        super().__init__()
        self.lambda_kl = lambda_kl
        self.annealing_step = annealing_step

    def _kl_dirichlet_uniform(self, alpha):
        K = alpha.shape[-1]
        S = alpha.sum(dim=-1, keepdim=True)
        log_ratio = (
            torch.lgamma(S)
            - torch.lgamma(alpha).sum(dim=-1, keepdim=True)
            - torch.lgamma(torch.tensor(float(K), device=alpha.device, dtype=alpha.dtype))
        )
        cross = ((alpha - 1.0) * (torch.digamma(alpha) - torch.digamma(S))).sum(dim=-1, keepdim=True)
        return (log_ratio + cross).squeeze(-1).clamp(min=0.0)

    def forward(self, score_pos, score_neg, alpha_v, alpha_t, epoch):
        bpr_loss = -F.logsigmoid(score_pos - score_neg).mean()
        kl_v = self._kl_dirichlet_uniform(alpha_v).mean()
        kl_t = self._kl_dirichlet_uniform(alpha_t).mean()
        lam = self.lambda_kl * min(1.0, epoch / max(1, self.annealing_step))
        total = bpr_loss + lam * (kl_v + kl_t)
        return {'total': total, 'bpr': bpr_loss.detach(),
                'kl_v': kl_v.detach(), 'kl_t': kl_t.detach(), 'lambda_kl': lam}

In [4]:

class Net(nn.Module):
    """
    MMGCN fiel al repo (un GCN por modalidad, 3 capas, preference, id_embedding)
    con fusión evidencial (EvidentialFusionLayer) y loss BPR + KL + L2 reg.
    """
    def __init__(self, img_feat, text_feat, edge_index,
                 num_user, num_item, aggr_mode, concate, has_id,
                 reg_weight, dim_x, evidence_dim, lambda_kl, annealing_step, device):
        super().__init__()
        self.num_user = num_user
        self.num_item = num_item
        self.reg_weight = reg_weight
        self.device = device

        # Adyacencia bidireccional normalizada (igual que nb14)
        N = num_user + num_item
        ei = np.array(edge_index)
        rows = np.concatenate([ei[:, 0], ei[:, 1]])
        cols = np.concatenate([ei[:, 1], ei[:, 0]])
        deg = np.bincount(rows, minlength=N).astype(float)
        if aggr_mode == 'mean':
            vals = np.where(deg[rows] > 0, 1.0 / deg[rows], 0.0)
        else:
            vals = np.ones(len(rows), dtype=np.float32)
        idx = torch.LongTensor(np.stack([rows, cols]))
        v = torch.FloatTensor(vals)
        self.register_buffer('adj', torch.sparse_coo_tensor(idx, v, (N, N)).coalesce())

        self.register_buffer('img_feat',  img_feat.float())
        self.register_buffer('text_feat', text_feat.float())

        # Un GCN por modalidad (igual que nb14)
        self.img_gcn = GCN(num_user, num_item, img_feat.shape[1],
                           dim_x, aggr_mode, concate, has_id, dim_latent=DIM_LATENT_V)
        self.text_gcn = GCN(num_user, num_item, text_feat.shape[1],
                            dim_x, aggr_mode, concate, has_id, dim_latent=None)

        self.id_embedding = nn.Parameter(
            nn.init.xavier_normal_(torch.rand(num_user + num_item, dim_x))
        )

        # EDL: fusión evidencial + loss (reemplaza el promedio de nb14)
        self.fusion = EvidentialFusionLayer(dim_x, evidence_dim)
        self.criterion = EvidentialLoss(lambda_kl, annealing_step)

    def _encode(self):
        """Propaga ambas GCNs y devuelve representaciones separadas [N, dim]."""
        img_rep = self.img_gcn(self.img_feat,  self.id_embedding, self.adj)
        text_rep = self.text_gcn(self.text_feat, self.id_embedding, self.adj)
        return img_rep, text_rep

    def forward(self):
        """Para recomendación: devuelve embeddings fusionados [N, dim]."""
        img_rep, text_rep = self._encode()
        fused, _, _ = self.fusion(img_rep, text_rep)
        return fused

    def loss(self, user_tensor, item_tensor, epoch):
        user_tensor = user_tensor.view(-1)   # [2B]: [u,u,u,...]
        item_tensor = item_tensor.view(-1)   # [2B]: [pos,neg,pos,...]

        img_rep, text_rep = self._encode()   # [N, dim] cada uno

        # Índices de usuario e ítem del mini-batch
        u_img = img_rep[user_tensor]
        u_txt = text_rep[user_tensor]
        i_img = img_rep[item_tensor]
        i_txt = text_rep[item_tensor]

        # Fusión evidencial: usuarios y pares (pos/neg) de ítems
        u_fused, _, _ = self.fusion(u_img, u_txt)      # [2B, dim]
        i_fused, av, at = self.fusion(i_img, i_txt)    # [2B, dim], [2B, K], [2B, K]

        # BPR scores: [B, 2] → col0=pos, col1=neg
        dot = torch.sum(u_fused * i_fused, dim=1).view(-1, 2)
        score_pos = dot[:, 0]
        score_neg = dot[:, 1]

        # BPR + KL con annealing (EvidentialLoss de nb12)
        losses = self.criterion(score_pos, score_neg, av, at, epoch)

        # L2 reg sobre id_embedding y preference (igual que nb14)
        reg_emb = (
            (self.id_embedding[user_tensor]**2 + self.id_embedding[item_tensor]**2).mean()
            + (self.img_gcn.preference**2).mean()
        )
        total = losses['total'] + self.reg_weight * reg_emb
        return total, losses

In [5]:
# ── TrainingDataset: idéntico a nb14 ─────────────────────────────────────────

class TrainingDataset(Dataset):
    """
    Fiel al TrainingDataset del repo original.
    Cada sample: ([user, user], [pos_item, neg_item]).
    """
    def __init__(self, num_user, num_item, user_item_dict, pos_edges):
        self.pos_edges = pos_edges
        self.user_item_dict = user_item_dict
        self.all_items = list(range(num_user, num_user + num_item))

    def __len__(self):
        return len(self.pos_edges)

    def __getitem__(self, index):
        user, pos_item = self.pos_edges[index]
        seen = self.user_item_dict.get(user, set())
        neg_item = random.choice(self.all_items)
        while neg_item in seen:
            neg_item = random.choice(self.all_items)
        return torch.LongTensor([user, user]), torch.LongTensor([pos_item, neg_item])

In [6]:

class MMGCN_EDL:
    """MMGCN (fiel al repo) con fusión evidencial."""

    def __init__(self, text_embeddings, image_embeddings, embed_dim=64, device='cpu'):
        common = sorted(set(text_embeddings) & set(image_embeddings))
        self._item_ids = common
        self._item_idx = {b: i for i, b in enumerate(common)}
        self.num_item = len(common)

        self._img_feat = torch.FloatTensor(np.stack([image_embeddings[b] for b in common]))
        self._text_feat = torch.FloatTensor(np.stack([text_embeddings[b] for b in common]))

        self._embed_dim = embed_dim
        self.device = device
        self._net = None
        self._user_idx = None
        self.num_user = None
        self._emb_cache = None

        print(f'MMGCN_EDL: {self.num_item:,} items | '
              f'image={self._img_feat.shape[1]}d | text={self._text_feat.shape[1]}d')

    def _build_structures(self, train_reviews):
        filtered = train_reviews[
            train_reviews['business_id'].isin(self._item_idx)
        ].copy()

        all_users = filtered['user_id'].unique()
        self._user_idx = {u: i for i, u in enumerate(all_users)}
        self.num_user = len(all_users)

        u_nodes = filtered['user_id'].map(self._user_idx).values
        i_nodes = filtered['business_id'].map(self._item_idx).values + self.num_user
        all_edges = list(zip(u_nodes.tolist(), i_nodes.tolist()))

        user_item_dict = defaultdict(set)
        for u, i in all_edges:
            user_item_dict[u].add(i)

        liked = filtered[filtered['stars'] >= LIKE_THRESHOLD]
        lu = liked['user_id'].map(self._user_idx).values
        li = liked['business_id'].map(self._item_idx).values + self.num_user
        liked_edges = list(zip(lu.tolist(), li.tolist())) or all_edges

        print(f'  Usuarios: {self.num_user:,} | '
              f'Aristas grafo: {len(all_edges):,} | '
              f'Positivos (≥{LIKE_THRESHOLD}★): {len(liked_edges):,}')
        return all_edges, dict(user_item_dict), liked_edges

    def fit(self, train_reviews,
            epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
            batch_size=BATCH_SIZE, evidence_dim=EVIDENCE_DIM,
            lambda_kl=LAMBDA_KL, annealing_step=ANNEALING_STEP):
        torch.manual_seed(42)
        random.seed(42)
        self._emb_cache = None

        all_edges, user_item_dict, liked_edges = self._build_structures(train_reviews)

        self._net = Net(
            self._img_feat, self._text_feat, all_edges,
            self.num_user, self.num_item,
            aggr_mode=AGGR_MODE, concate=CONCATE, has_id=HAS_ID,
            reg_weight=weight_decay, dim_x=self._embed_dim,
            evidence_dim=evidence_dim,
            lambda_kl=lambda_kl, annealing_step=annealing_step,
            device=self.device,
        ).to(self.device)

        dataset = TrainingDataset(self.num_user, self.num_item, user_item_dict, liked_edges)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        optimizer = torch.optim.Adam(self._net.parameters(), lr=lr)

        print(f'BPR+EDL: {len(dataset):,} positivos | {epochs} epochs | bs={batch_size}')

        for epoch in range(1, epochs + 1):
            self._net.train()
            s_total = s_bpr = s_kl_v = s_kl_t = steps = 0
            for u_t, i_t in loader:
                u_t, i_t = u_t.to(self.device), i_t.to(self.device)
                optimizer.zero_grad()
                total, ldict = self._net.loss(u_t, i_t, epoch)
                total.backward()
                optimizer.step()
                s_total += total.item()
                s_bpr += ldict['bpr'].item()
                s_kl_v += ldict['kl_v'].item()
                s_kl_t += ldict['kl_t'].item()
                steps += 1
            if epoch % 10 == 0:
                lam = ldict['lambda_kl']
                print(f'  Epoch {epoch:3d}/{epochs} | '
                      f'loss: {s_total/steps:.4f} | bpr: {s_bpr/steps:.4f} | '
                      f'KL_v: {s_kl_v/steps:.4f} | KL_t: {s_kl_t/steps:.4f} | '
                      f'λ={lam:.3f}')

        self._net.eval()
        return self

    def recommend(self, user_id, train_reviews, top_k=10):
        if self._net is None or user_id not in self._user_idx:
            return []
        seen = set(train_reviews[train_reviews['user_id'] == user_id]['business_id'])
        cands = [b for b in self._item_ids if b not in seen]
        if not cands:
            return []

        if self._emb_cache is None:
            with torch.no_grad():
                self._emb_cache = self._net.forward().detach()

        u_node = self._user_idx[user_id]
        c_nodes = [self.num_user + self._item_idx[b] for b in cands]
        u_f = self._emb_cache[u_node].unsqueeze(0)
        c_f = self._emb_cache[c_nodes]
        scores = (u_f * c_f).sum(-1).cpu().numpy()

        top = np.argsort(scores)[::-1][:top_k]
        return [cands[i] for i in top]

## Datos y embeddings

In [7]:
reviews = pd.read_csv('../data/processed/reviews.csv', parse_dates=['date'])
train_reviews, test_reviews = temporal_train_test_split(reviews, test_fraction=0.2)

text_embeddings = TextFeatureExtractor.load('text_embeddings.npz')
image_embeddings = CLIPFeatureExtractor.load('clip_embeddings.npz')

print(f'Text: {len(text_embeddings):,} | Image (CLIP): {len(image_embeddings):,}')
print(f'Overlap: {len(set(text_embeddings) & set(image_embeddings)):,} items con ambas modalidades')

Train: 83256 reviews | Test: 17191 reviews
Text: 1,151 | Image (CLIP): 3,824
Overlap: 1,151 items con ambas modalidades


## Entrenar MMGCN + EDL

In [8]:
model = MMGCN_EDL(
    text_embeddings, image_embeddings,
    embed_dim=EMBED_DIM, device=DEVICE,
)
model.fit(
    train_reviews,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    evidence_dim=EVIDENCE_DIM,
    lambda_kl=LAMBDA_KL,
    annealing_step=ANNEALING_STEP,
)

MMGCN_EDL: 1,151 items | image=512d | text=384d
  Usuarios: 10,490 | Aristas grafo: 83,256 | Positivos (≥4.0★): 62,155
BPR+EDL: 62,155 positivos | 50 epochs | bs=1024
  Epoch  10/50 | loss: 0.1334 | bpr: 0.1326 | KL_v: 0.0029 | KL_t: 0.0046 | λ=0.100
  Epoch  20/50 | loss: 0.1106 | bpr: 0.1105 | KL_v: 0.0005 | KL_t: 0.0010 | λ=0.100
  Epoch  30/50 | loss: 0.1019 | bpr: 0.1019 | KL_v: 0.0003 | KL_t: 0.0003 | λ=0.100
  Epoch  40/50 | loss: 0.0935 | bpr: 0.0934 | KL_v: 0.0001 | KL_t: 0.0002 | λ=0.100
  Epoch  50/50 | loss: 0.0843 | bpr: 0.0842 | KL_v: 0.0001 | KL_t: 0.0001 | λ=0.100


## Evaluación

In [9]:
metrics = evaluate_model(
    lambda uid, top_k: model.recommend(uid, train_reviews, top_k),
    test_reviews, train_reviews, k_values=[5, 10, 20]
)
print('MMGCN + EDL metrics:')
print(metrics.round(4))

MMGCN + EDL metrics:
    precision  recall    ndcg
K                            
5      0.0281  0.0940  0.0668
10     0.0229  0.1522  0.0868
20     0.0179  0.2353  0.1097


## Guardar resultados

In [10]:
os.makedirs(RESULTS_DIR, exist_ok=True)
metrics.to_csv(f'{RESULTS_DIR}/metrics.csv')
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump({
        'model': 'MMGCN_EDL',
        'embed_dim': EMBED_DIM,
        'dim_latent_v': DIM_LATENT_V,
        'evidence_dim': EVIDENCE_DIM,
        'aggr_mode': AGGR_MODE,
        'concate': CONCATE,
        'has_id': HAS_ID,
        'epochs': EPOCHS,
        'lr': LR,
        'weight_decay': WEIGHT_DECAY,
        'lambda_kl': LAMBDA_KL,
        'annealing_step': ANNEALING_STEP,
        'batch_size': BATCH_SIZE,
        'like_threshold': LIKE_THRESHOLD,
        'image_model': 'CLIP (512d)',
        'text_model': 'all-MiniLM-L6-v2',
        'loss': 'BPR + KL-EDL + L2 reg',
        'fusion': 'EvidentialFusionLayer (Subjective Logic)',
        'gcn_layers': 3,
        'n_items_multimodal': model.num_item,
        'n_users': model.num_user,
    }, f, indent=2)
print(f'Saved -> results/{MODEL_NAME}/')

Saved -> results/mmgcn_edl/
